# Trajectory failure analysis

Visualizza (grafici + tabelle) l'output di `scripts/trajectory_failure_analysis.py`:
`trajectory_runs.csv` — una riga per run, con un **failure mode** mutuamente esclusivo,
il **failure stage** (node della pipeline in cui si manifesta) e le **metriche di
traiettoria** (tool usage, duplicati, iterazioni, file letti, ...).

Contenuto (ogni sezione mostra grafico + tabella dei numeri):
1. Caricamento dati
2. Tabella failure mode × modello (conteggi + %, con export LaTeX)
3. Stacked bar dei failure mode per modello + tabella
4. Facet per solver (come cambia il profilo di fallimento con l'orchestrazione) + tabella
5. Metriche di traiettoria: run corretti vs sbagliati — media per-run **e** totali
6. `diff_kind` delle wrong answer (indizio reasoning failure vs artifact di confronto GT)
7. `failure_stage`: dove si rompe la pipeline (heatmap failure_mode × stadio) + focus empty_response

Per (ri)generare il CSV, dalla root del repo:
```
python scripts/trajectory_failure_analysis.py local_models [altre_dir ...] --out-dir trajectory_analysis
```

In [ ]:
from pathlib import Path

# --- IMPOSTA QUI LA SORGENTE -------------------------------------------------
TRAJ_SOURCE = "trajectory_runs.csv"   # prodotto da scripts/trajectory_failure_analysis.py
OUTDIR = Path("figures")              # dove salvare PDF/PNG
SAVE_PNG_TOO = True
# -----------------------------------------------------------------------------
OUTDIR.mkdir(parents=True, exist_ok=True)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Stile identico a notebooks/paper_plots.ipynb
sns.set_theme(style="whitegrid", context="paper", font_scale=1.15)
plt.rcParams.update({
    "figure.dpi": 110,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "axes.titleweight": "semibold",
})

SOLVER_NAMES = {
    "BulkProcessingSolverFromFiles": "bulk",
    "BulkReactSolverFromFiles": "bulk_react",
    "PlannerAgentSolver": "planner_agent",
    "PlannerAgentSolverWithNetwork": "planner_agent_net",
    "StrategicAgentSolver": "guided_retrieval_agent",
}
SOLVER_ORDER = ["bulk", "bulk_react", "planner_agent", "guided_retrieval_agent", "planner_agent_net"]


def short_model(m: str) -> str:
    return m.split("(")[0].strip()


def short_lab(l: str) -> str:
    return l.replace("lab_", "")


def savefig(fig, name: str):
    pdf_path = OUTDIR / f"{name}.pdf"
    fig.savefig(pdf_path)
    if SAVE_PNG_TOO:
        fig.savefig(OUTDIR / f"{name}.png")
    print("saved:", pdf_path)


## 1. Caricamento dati

In [ ]:
df = pd.read_csv(TRAJ_SOURCE)
df["solver"] = df["solver"].map(lambda s: SOLVER_NAMES.get(s, s))
df["model_short"] = df["model"].map(short_model)
df["lab_short"] = df["lab_name"].map(short_lab)

# Ordine canonico e palette dei failure mode (correct in verde, il resto a contrasto)
MODE_ORDER = ["correct", "wrong_answer", "empty_response", "unparsed_tool_call",
              "tool_hallucination", "tool_call_syntax", "malformed_json",
              "schema_violation", "loop_exhaustion", "timeout",
              "environment_error", "crash_other"]
MODE_PALETTE = {
    "correct": "#1D9E75", "wrong_answer": "#D85A30", "empty_response": "#BA7517",
    "unparsed_tool_call": "#D4537E", "tool_hallucination": "#7F77DD",
    "tool_call_syntax": "#639922", "malformed_json": "#378ADD",
    "schema_violation": "#13426E", "loop_exhaustion": "#888780",
    "timeout": "#555555", "environment_error": "#222222", "crash_other": "#aaaaaa",
}
modes_present = [m for m in MODE_ORDER if m in set(df["failure_mode"])]
models_present = sorted(df["model_short"].unique())
solvers_present = [s for s in SOLVER_ORDER if s in set(df["solver"])]

print(f"{len(df)} runs | {len(models_present)} modelli | {len(solvers_present)} solver")
print("Failure mode presenti:", modes_present)
df.head(3)


## 1b. Esiti per modello × solver — corretti / sbagliati / invalid run

Riepilogo a tre vie per ogni coppia **modello × solver** (conteggi **e** percentuali per-coppia),
con una riga **`ALL`** in coda a ogni modello = totale aggregato su tutti i solver (in grassetto):

- **correct** — `failure_mode == "correct"` (risposta giusta).
- **wrong** — `failure_mode == "wrong_answer"` (risposta strutturalmente valida ma sbagliata nel contenuto).
- **invalid** — *tutto il resto* (`empty_response`, `unparsed_tool_call`, `tool_hallucination`,
  `tool_call_syntax`, `malformed_json`, `schema_violation`, `loop_exhaustion`, `timeout`,
  `environment_error`, `crash_other`): run che **non** hanno prodotto una risposta utilizzabile.

In [ ]:
# Esito a tre vie per ogni run: correct / wrong / invalid.
# correct = risposta giusta; wrong = wrong_answer (valida ma sbagliata);
# invalid = ogni altro failure_mode (run senza risposta utilizzabile).
OUTCOME_ORDER = ["correct", "wrong", "invalid"]


def to_outcome(mode: str) -> str:
    if mode == "correct":
        return "correct"
    if mode == "wrong_answer":
        return "wrong"
    return "invalid"


tmp = df.assign(outcome=df["failure_mode"].map(to_outcome))

oc = (tmp.groupby(["model_short", "solver", "outcome"]).size()
      .unstack(fill_value=0).reindex(columns=OUTCOME_ORDER, fill_value=0))
oc = oc.reindex(pd.MultiIndex.from_product(
    [models_present, solvers_present], names=["model_short", "solver"]), fill_value=0)

# Riga di totale per modello (solver = "ALL", aggregando tutti i solver), inserita
# in coda al blocco di ogni modello.
tot = (tmp.groupby(["model_short", "outcome"]).size()
       .unstack(fill_value=0).reindex(columns=OUTCOME_ORDER, fill_value=0)
       .reindex(models_present, fill_value=0))
tot.index = pd.MultiIndex.from_arrays(
    [tot.index, ["ALL"] * len(tot)], names=["model_short", "solver"])
oc = pd.concat([oc, tot]).sort_index(
    level="model_short", sort_remaining=False, kind="stable")

oc_pct = oc.div(oc.sum(axis=1).replace(0, np.nan), axis=0).mul(100).fillna(0)

# Tabella combinata: percentuale per-coppia con conteggio tra parentesi, + colonna n.
tbl1b = oc_pct.round(1).astype(str) + "% (" + oc.astype(str) + ")"
tbl1b.insert(0, "n", oc.sum(axis=1))


def _highlight_total(row):
    # Evidenzia in grassetto le righe di totale per modello (solver == "ALL").
    is_total = row.name[1] == "ALL"
    return ["font-weight: bold" if is_total else "" for _ in row]


display(
    oc_pct.style
    .background_gradient(cmap="Greens", subset=["correct"])
    .background_gradient(cmap="Oranges", subset=["wrong"])
    .background_gradient(cmap="Reds", subset=["invalid"])
    .apply(_highlight_total, axis=1)
    .format("{:.1f}%")
)
display(tbl1b)

# Export LaTeX
print(tbl1b.to_latex(column_format="ll" + "r" * (len(OUTCOME_ORDER) + 1)))


## 2. Failure mode × modello — tabella

In [ ]:
counts = (df.groupby(["model_short", "failure_mode"]).size()
          .unstack(fill_value=0).reindex(columns=modes_present, fill_value=0))
pct = counts.div(counts.sum(axis=1), axis=0).mul(100)

display(
    pct.style
    .background_gradient(cmap="Reds", axis=None, subset=[m for m in modes_present if m != "correct"])
    .background_gradient(cmap="Greens", subset=["correct"])
    .format("{:.1f}%")
)

# Export LaTeX: percentuale con conteggio tra parentesi
latex_tbl = pct.round(1).astype(str) + "% (" + counts.astype(str) + ")"
print(latex_tbl.to_latex(column_format="l" + "r" * len(modes_present)))


## 3. Failure mode per modello — stacked bar

In [ ]:
pv = pct.loc[models_present]
fig, ax = plt.subplots(figsize=(8.5, 4.2))
bottom = np.zeros(len(pv))
for mode in modes_present:
    ax.bar(pv.index, pv[mode], bottom=bottom, label=mode,
           color=MODE_PALETTE[mode], width=0.72, edgecolor="white", linewidth=0.4)
    bottom += pv[mode].values
ax.set_ylabel("% of runs")
ax.set_ylim(0, 100)
ax.legend(fontsize=7.5, ncol=2, loc="lower left", framealpha=0.9)
ax.tick_params(axis="x", rotation=30)
for lbl in ax.get_xticklabels():
    lbl.set_ha("right")
fig.tight_layout()
savefig(fig, "trajectory_failure_modes_by_model")
plt.show()


In [ ]:
# Tabella dei numeri dietro lo stacked bar: conteggi + percentuali per failure mode.
# (stessa aggregazione del grafico sopra: `counts`/`pct` per modello)
tbl3 = pct.round(1).astype(str) + "% (" + counts.astype(str) + ")"
tbl3.insert(0, "n", counts.sum(axis=1))
display(tbl3)


## 4. Facet per solver

Come cambia il profilo di fallimento di ogni modello al variare dell'orchestrazione
(es. llama3.1: wrong answer "normali" sotto bulk, collasso in empty/unparsed tool call
sotto i solver agentici).

In [ ]:
fig, axes = plt.subplots(1, len(solvers_present), figsize=(3.4 * len(solvers_present), 4.0),
                         sharey=True)
for ax, solver in zip(np.atleast_1d(axes), solvers_present):
    sub = df[df["solver"] == solver]
    c = (sub.groupby(["model_short", "failure_mode"]).size()
         .unstack(fill_value=0).reindex(columns=modes_present, fill_value=0)
         .reindex(models_present, fill_value=0))
    p = c.div(c.sum(axis=1).replace(0, np.nan), axis=0).mul(100).fillna(0)
    bottom = np.zeros(len(p))
    for mode in modes_present:
        ax.bar(p.index, p[mode], bottom=bottom, label=mode,
               color=MODE_PALETTE[mode], width=0.7, edgecolor="white", linewidth=0.3)
        bottom += p[mode].values
    ax.set_title(solver, fontsize=10)
    ax.set_ylim(0, 100)
    ax.tick_params(axis="x", rotation=90, labelsize=7.5)
axes_flat = np.atleast_1d(axes)
axes_flat[0].set_ylabel("% of runs")
axes_flat[-1].legend(fontsize=7, loc="center left", bbox_to_anchor=(1.02, 0.5))
fig.tight_layout()
savefig(fig, "trajectory_failure_modes_by_solver")
plt.show()


In [ ]:
# Tabella dei numeri dietro il facet: conteggi + percentuali per (solver, modello).
# Stesse aggregazioni delle facet sopra, impilate in un'unica tabella indicizzata
# da (solver, model_short). Le percentuali sono per-riga (= per coppia solver-modello).
rows4 = []
for solver in solvers_present:
    sub = df[df["solver"] == solver]
    c = (sub.groupby(["model_short", "failure_mode"]).size()
         .unstack(fill_value=0).reindex(columns=modes_present, fill_value=0)
         .reindex(models_present, fill_value=0))
    p = c.div(c.sum(axis=1).replace(0, np.nan), axis=0).mul(100).fillna(0)
    cell = p.round(1).astype(str) + "% (" + c.astype(str) + ")"
    cell.insert(0, "n", c.sum(axis=1))
    cell.index = pd.MultiIndex.from_product([[solver], cell.index],
                                            names=["solver", "model_short"])
    rows4.append(cell)

tbl4 = pd.concat(rows4)
display(tbl4)


## 5. Metriche di traiettoria — corretti vs sbagliati

Solo `planner_agent` (l'unico solver con tool). Qui si vede il *come* falliscono:
rnj-1 fa thrashing (tante chiamate ripetute che falliscono), llama non chiama mai i tool,
Ministral sbaglia pur leggendo più file dei run corretti (reasoning, non retrieval).

**Nota su media vs totale:** la prima tabella è la **media per-run** di ogni metrica
(`.mean()`, *on average* per ciascun run del gruppo modello × esito). La seconda è il
**totale** (`.sum()`, somma su tutti i run del gruppo). I totali vanno letti insieme
alla colonna `n` (numero di run nel gruppo): un gruppo più numeroso accumula totali più
alti anche a parità di comportamento per-run.

In [ ]:
B_METRICS = ["n_tool_calls", "n_tool_errors", "n_duplicate_calls",
             "n_planner_iterations", "n_files_read", "n_empty_llm_outputs"]

planner = (df[df["solver"] == "planner_agent"]
           .assign(esito=lambda d: d["correct"].map({True: "correct", False: "wrong"})))
grp = planner.groupby(["model_short", "esito"])

# --- Media per-run (on average) ---------------------------------------------
tbl_mean = grp[B_METRICS].mean().round(2)
print("== Media per-run (on average) ==")
display(tbl_mean.style.background_gradient(cmap="Oranges", axis=0).format("{:.2f}"))
print(tbl_mean.to_latex(float_format="%.2f"))

# --- Totali (somma su tutti i run del gruppo) -------------------------------
tbl_sum = grp[B_METRICS].sum()
tbl_sum.insert(0, "n", grp.size())   # numero di run nel gruppo, per interpretare i totali
print("\n== Totale (somma sul gruppo) ==")
display(tbl_sum.style.background_gradient(cmap="Purples", axis=0, subset=B_METRICS).format("{:.0f}"))
print(tbl_sum.to_latex())


## 6. Wrong answer: forma del diff con la ground truth

`diff_kind` (in `scripts/trajectory_failure_analysis.py::diff_kind`) classifica la
*forma* del `verification_diff` dei soli run `wrong_answer`. Di default il diff è
l'output di `DeepDiff(ground_truth, response, ignore_order=True)`
(`questions/base_question.py::verify`); la classe di chiavi DeepDiff presenti determina
il `diff_kind`:

- **values_changed_only** — il diff contiene *solo* `values_changed` / `type_changes`:
  **stessa struttura** della ground truth (stessi item e chiavi), **valori di foglia
  diversi**. Candidato sia per *artifact di confronto GT* (naming/formato) sia per
  errore di reasoning sul valore. Es.: numero di nodi sbagliato, ping yes/no errato.
- **missing_items / extra_items** — solo chiavi `*removed*` / solo `*added*`: risposta
  incompleta (item mancanti) / con elementi inventati.
- **mixed** — added + removed + changed insieme: reasoning failure strutturale piena.
- **custom** — chiave non-DeepDiff nel diff. ⚠️ Nei dati corrisponde **esclusivamente
  alle domande traceroute**, l'unica question con `verify()` di dominio
  (`questions/traceroute.py`) che ritorna `{"traceroute": ...}` quando il percorso è
  irraggiungibile o il forwarding path non è valido. **Non** è un output
  strutturalmente malformato: significa "il validatore del traceroute ha bocciato il
  path".

In [ ]:
wrong = df[df["failure_mode"] == "wrong_answer"]
dk = (wrong.groupby(["model_short", "diff_kind"]).size()
      .unstack(fill_value=0))
dk_pct = dk.div(dk.sum(axis=1), axis=0).mul(100)

display(dk_pct.style.background_gradient(cmap="Blues", axis=None).format("{:.1f}%"))

fig, ax = plt.subplots(figsize=(8, 3.8))
bottom = np.zeros(len(dk_pct))
palette = dict(zip(dk_pct.columns, sns.color_palette("colorblind", n_colors=len(dk_pct.columns))))
for kind in dk_pct.columns:
    ax.bar(dk_pct.index, dk_pct[kind], bottom=bottom, label=kind, color=palette[kind],
           width=0.7, edgecolor="white", linewidth=0.4)
    bottom += dk_pct[kind].values
ax.set_ylabel("% delle wrong answer")
ax.set_ylim(0, 100)
ax.legend(fontsize=8, ncol=2)
ax.tick_params(axis="x", rotation=30)
for lbl in ax.get_xticklabels():
    lbl.set_ha("right")
fig.tight_layout()
savefig(fig, "trajectory_wrong_answer_diff_kind")
plt.show()


## 7. Distribuzione del fallimento lungo la pipeline

La colonna `failure_stage` (CSV, prodotta da `failure_stage()` nello script) indica il
**node in cui il fallimento si manifesta**. Serve perché `last_node` è *sempre*
`structured_output` (il node terminale di ogni run) e quindi nasconde dove le cose si
rompono davvero. Regola di attribuzione:

- `empty_response` → node dell'(ultimo) `llm_output` vuoto (stadio di reasoning /
  stesura risposta / estrazione strutturata finale);
- `tool_hallucination` / `unparsed_tool_call` → node dell'`llm_output` che ha inventato
  un tool o emesso una tool call come testo;
- crash, `wrong_answer`, `loop_exhaustion` → ultimo node eseguito (`last_node`);
- `correct` → `answered`.

### Tassonomia dei node per solver
| solver | catena di node |
|---|---|
| Strategic (`guided_retrieval_agent`) | `strategy_classifier` → `context_assembler` → `analyst` → `structured_output` |
| Planner (`planner_agent`) | `planner` ↔ `tools` (loop) → `validation` → `final_answer` → `structured_output` |
| Bulk (`bulk`) | `llm_solver` → `structured_output` |
| BulkReact (`bulk_react`) | `reason` ↔ `act` → `structured_output` |

`structured_output` è il node di estrazione Pydantic finale, presente in **ogni** solver.

### Risultati chiave (riferimento per scrivere la section)
- **Le empty response nascono a monte, quasi mai allo `structured_output`.** Si
  localizzano nello stadio di reasoning / stesura risposta: Strategic→`analyst` (515,
  mai con tool — non ha tool), Bulk→`llm_solver` (105), BulkReact→`act` (11),
  Planner→`final_answer` (672). Nel Planner **500/672 emettono vuoto senza aver mai
  chiamato un tool** (si arrendono subito), **172/672 dopo aver chiamato i tool** (la
  parte agentica gira, ma la risposta finale è vuota). Lo `structured_output` con
  fallback quasi sempre produce *qualcosa*, quindi non è lì che muoiono.
- **I fallimenti di estrazione strutturata** (`malformed_json`, `schema_violation`,
  `crash_other`) si concentrano su `structured_output`, con una piccola coda su
  `strategy_classifier` (anche il classifier dello Strategic fa una chiamata strutturata).
- **I fallimenti tool-related** (`tool_call_syntax`, `unparsed_tool_call`,
  `tool_hallucination`) sono **esclusivi del `planner`**: solo il Planner solver li
  esibisce, essendo l'unico con loop di tool.
- **`wrong_answer`** raggiunge per definizione `structured_output` con risposta completa
  e parseabile → è un fallimento di *contenuto* (reasoning), non di *posizione* nella
  pipeline.

In [ ]:
# Heatmap failure_mode × failure_stage: dove si manifesta ogni modo di fallimento.
# Numero in cella = run; colore = % della riga (cioè % di quel failure_mode che cade
# in quello stadio). 'correct' escluso (cade tutto su 'answered').
STAGE_ORDER = ["strategy_classifier", "context_assembler", "llm_solver", "reason",
               "act", "analyst", "planner", "tools", "validation", "final_answer",
               "structured_output", "answered"]
stages_present = [s for s in STAGE_ORDER if s in set(df["failure_stage"])]
fail_modes = [m for m in MODE_ORDER if m != "correct" and m in set(df["failure_mode"])]

ct = (df[df["failure_mode"] != "correct"]
      .groupby(["failure_mode", "failure_stage"]).size()
      .unstack(fill_value=0)
      .reindex(index=fail_modes, columns=stages_present, fill_value=0))
ct_pct = ct.div(ct.sum(axis=1).replace(0, np.nan), axis=0).mul(100).fillna(0)

annot = ct.astype(int).astype(str).where(ct > 0, "")
fig, ax = plt.subplots(figsize=(0.95 * len(stages_present) + 2.5, 0.5 * len(ct) + 1.6))
sns.heatmap(ct_pct, annot=annot, fmt="", cmap="rocket_r", vmin=0, vmax=100,
            cbar_kws={"label": "% del failure_mode (riga)"},
            linewidths=0.5, linecolor="white", ax=ax)
ax.set_xlabel("failure_stage (node della pipeline)")
ax.set_ylabel("failure_mode")
ax.set_title("Dove si manifesta ogni failure mode lungo la pipeline\n"
             "(numero = run, colore = % della riga)")
plt.setp(ax.get_xticklabels(), rotation=35, ha="right")
fig.tight_layout()
savefig(fig, "trajectory_failure_stage_heatmap")
plt.show()

# Tabella dei numeri: conteggi + % per riga, con totale 'n'
tbl7 = ct_pct.round(1).astype(str) + "% (" + ct.astype(str) + ")"
tbl7.insert(0, "n", ct.sum(axis=1))
display(tbl7)


In [ ]:
# Focus empty_response: per ogni stadio, quante empty arrivano DOPO aver chiamato i tool
# vs senza alcuna tool call. Distingue "il modello si arrende subito" (no_tool_calls) da
# "la parte agentica gira ma la risposta finale è vuota" (tool_calls>0).
er = df[df["failure_mode"] == "empty_response"].copy()
er["had_tools"] = er["n_tool_calls"].gt(0).map({True: "tool_calls>0", False: "no_tool_calls"})

foc = pd.crosstab(er["failure_stage"], er["had_tools"])
for col in ("no_tool_calls", "tool_calls>0"):
    if col not in foc.columns:
        foc[col] = 0
foc = foc[["no_tool_calls", "tool_calls>0"]]
foc["totale"] = foc.sum(axis=1)
# solver dominante per stadio (le empty sono di fatto solver-specifiche)
foc.insert(0, "solver", er.groupby("failure_stage")["solver"].agg(lambda s: s.value_counts().index[0]))
foc = foc.reindex([s for s in STAGE_ORDER if s in foc.index])
print(f"empty_response: {len(er)} run totali")
display(foc)
